# S1 - Taxonomy and Deduplication

This stage audits the complete taxonomy and duplicate leakage risk. The
target represents routing to historical CFPB form categories, not harm
prevalence or a market condition.

This notebook does not train a model or choose the final temporal split.

In [1]:
from pathlib import Path
import sys

import polars as pl

project_root = Path.cwd().resolve()
root_found = (project_root / 'pyproject.toml').is_file()
while not root_found and project_root != project_root.parent:
    project_root = project_root.parent
    root_found = (project_root / 'pyproject.toml').is_file()
if not root_found:
    raise RuntimeError('Could not find project root with pyproject.toml')
sys.path.insert(0, str(project_root / 'src'))

from consumer_complaint_intelligence.s1 import load_or_run_s1

parquet_path = project_root / 'dataset' / 'processed' / 'complaints.parquet'
spill_path = project_root / 'temp' / 'duckdb'
cache_path = project_root / 'temp' / 's1' / 's1_report.json'
result = load_or_run_s1(parquet_path, cache_path, spill_path)
report = result['report']
print(report['audit'], report['taxonomy_version'], result['cache']['status'])

S1_taxonomy_deduplication cfpb-product-family-v1.0.0 hit


## Summary Evidence

The tables below show aggregates only. Coverage and concentration by year
help separate form changes, process changes, and corpus composition. No
consumer text is displayed.

In [2]:
key_years = [2017, 2023, 2025, 2026]
family_year = pl.DataFrame(
    report['taxonomy']['coverage_by_year_product_family']
).filter(pl.col('year').is_in(key_years))
mapping = pl.DataFrame(report['taxonomy']['mapping']['classes'])
distribution = pl.DataFrame(
    report['taxonomy']['narrative_distribution_by_family']
)
duplicates = pl.DataFrame(
    [
        {'method': method, **values}
        for method, values in report['deduplication'][
            'summary_by_method'
        ].items()
    ]
)
display(family_year)
display(mapping)
display(distribution)
display(duplicates)
display(report['taxonomy']['concentration'])
display(report['gate'])

label,narrative_coverage_pct,narrative_rows,total_rows,year
str,f64,i64,i64,i64
"""cards_prepaid""",48.4967,11146,22983,2017
"""consumer_lending""",54.0878,5769,10666,2017
"""credit_reporting""",47.3008,42549,89954,2017
"""debt_collection""",49.2756,23571,47835,2017
"""deposit_accounts""",37.1285,7321,19718,2017
…,…,…,…,…
"""debt_credit_management""",34.8672,932,2673,2026
"""deposit_accounts""",35.1302,18678,53168,2026
"""money_services""",36.892,8411,22799,2026


mapping_status,narrative_rows,raw_product,total_rows
str,i64,str,i64
"""mapped""",1671821,"""Credit reporting or other pers…",11588299
"""mapped""",807499,"""Credit reporting, credit repai…",2163770
"""mapped""",440886,"""Debt collection""",1169692
"""mapped""",146267,"""Mortgage""",457001
"""mapped""",186692,"""Checking or savings account""",387135
…,…,…,…
"""mapped""",5437,"""Debt or credit management""",9836
"""mapped""",1745,"""Payday loan""",5528
"""mapped""",1497,"""Money transfers""",5354


family,narrative_rows,narrative_share_pct
str,i64,f64
"""credit_reporting""",2510907,65.4451
"""debt_collection""",440886,11.4914
"""cards_prepaid""",247005,6.438
"""deposit_accounts""",201575,5.2539
"""mortgage""",146267,3.8124
"""money_services""",122121,3.183
"""consumer_lending""",99573,2.5953
"""student_loan""",62596,1.6315
"""debt_credit_management""",5437,0.1417


method,duplicate_group_rows,groups_cross_date,groups_cross_family,groups_cross_issue,groups_cross_raw_product,groups_cross_year,redundant_rows,total_duplicate_groups
str,i64,i64,i64,i64,i64,i64,i64,i64
"""exact""",1499532,37460,2632,8556,4437,7154,1253436,246096
"""normalized""",1588463,38854,2865,9217,4972,7674,1323391,265072


{'largest_family': {'family': 'credit_reporting',
  'narrative_rows': 2510907,
  'narrative_share_pct': 65.4451},
 'largest_family_share_pct': 65.4451,
 'narrative_rows': 3836659,
 'top_3_share_pct': 83.3745}

{'configuration': {'minimum_class_rows': None,
  'minimum_class_years': None,
  'split_cut_date': None},
 'gate_status': 'BLOCKED',
 'next_stage_requirements': ['Review 2017 and August 2023 taxonomy regime boundaries.',
  'Quantify credit_reporting concentration by year and narrative coverage.',
  'Investigate 2025/2026 volume and process-integrity anomaly without automatic exclusion.',
  'Choose a cut date and minimum class criteria explicitly in S2.',
  'Construct group-aware temporal partitions using normalized fingerprints.'],
 'reasons': ['No temporal split cut date has been configured.',
  'Minimum per-class row and year criteria are not both configured.',
  'The 2017 and August 2023 form regimes require review before splitting.',
  'Credit-reporting concentration and the 2025/2026 anomaly require review.',
  'S1 does not select or seal a scientific temporal split.']}

## Methodological Interpretation

The official CFPB page states that Product, Sub-product, Issue, and
Sub-issue preserve the original form selection available when the
complaint was submitted. The field reference states that Issue depends on
Product; therefore global Issue is semantically unstable. This stage uses
`product_family + Issue` as a secondary key without automatic merging.

The 2017 and August 2023 changes, `credit_reporting` concentration, and
the 2025/2026 volume anomaly must inform split selection. 2026 is partial
and may reflect process or integrity changes. The June 2026 notice is a
review signal, not a rule to exclude data automatically.

In [3]:
display(report['taxonomy']['sources'])
display(report['deduplication']['policy'])
display(report['deduplication']['top_groups'].copy())

{'cfpb_august_2023_pdf': 'https://files.consumerfinance.gov/f/documents/cfpb_consumer_complaint_form_product_issue_options_August_2023_FINAL.pdf',
 'cfpb_database': 'https://www.consumerfinance.gov/data-research/consumer-complaints/',
 'cfpb_fields': 'https://cfpb.github.io/api/ccdb/fields.html',
 'cfpb_june_2026_notice': 'https://www.consumerfinance.gov/about-us/newsroom/the-cfpb-is-correcting-flaws-to-restore-integrity-and-utility-to-the-consumer-complaint-system/'}

{'collision_risk': 'MD5 collisions are theoretical but not impossible; confirm critical groups with a collision-safe follow-up.',
 'exact_text': 'MD5 plus raw text length inside DuckDB.',
 'future_metrics': ['all_text_operational', 'novel_text_purged'],
 'is_repeated': 'True when the normalized group has more than one row.',
 'modeling_group_id': 'Use normalized fingerprint plus normalized length.',
 'normalized_text': 'Lowercase, trim, and collapse whitespace only.',
 'raw_dataset_preserved': True,
 'templates_policy': 'Near-duplicates beyond this normalization remain a later audit.',
 'unique_complaint_id': 'Integrity check only; it is not text deduplication.'}

[{'distinct_families': 2,
  'distinct_issues': 6,
  'distinct_raw_products': 3,
  'distinct_years': 6,
  'fingerprint': '010bc79fee6d82032c52a5800ef47076',
  'fingerprint_length': 272,
  'first_date': '2021-10-07',
  'first_year': 2021,
  'group_rows': 24246,
  'last_date': '2026-07-16',
  'last_year': 2026,
  'method': 'exact',
  'method_rank': 2},
 {'distinct_families': 1,
  'distinct_issues': 4,
  'distinct_raw_products': 1,
  'distinct_years': 2,
  'fingerprint': '13e61a070d5f7343582efeae3b87c545',
  'fingerprint_length': 626,
  'first_date': '2024-02-29',
  'first_year': 2024,
  'group_rows': 5097,
  'last_date': '2025-12-26',
  'last_year': 2025,
  'method': 'exact',
  'method_rank': 19},
 {'distinct_families': 1,
  'distinct_issues': 2,
  'distinct_raw_products': 1,
  'distinct_years': 1,
  'fingerprint': '1d19ee6f1c5ba7112a2bca03ae96698b',
  'fingerprint_length': 213,
  'first_date': '2025-01-22',
  'first_year': 2025,
  'group_rows': 10087,
  'last_date': '2025-06-09',
  'last